# ============================================
# COMPARING OPEN-SOURCE EMBEDDING MODELS FOR RAG
# ============================================

This notebook compares three popular embedding models to see which performs best for pharmaceutical document retrieval.

You'll learn:
- How different embedding models produce different results
- How to measure retrieval time and quality
- How to choose the right model for your use case
# ============================================

In [ ]:
# ============================================
# STEP 1: Install Required Libraries
# ============================================
#
# WHAT WE'RE DOING:
# Installing packages for RAG and embedding models:
# - llama-index: Core RAG framework
# - llama-index-embeddings-huggingface: Access to HuggingFace models
# - pymupdf: PDF text extraction
# - nest_asyncio: Handles async event loops in Colab
#
# WHY THIS MATTERS:
# These libraries let us test multiple embedding models easily.
#
# WHAT YOU'LL SEE:
# Installation progress (the -q flag keeps output minimal)
# ============================================

!pip install -q llama-index llama-index-embeddings-huggingface pymupdf
!pip install -q nest_asyncio

In [ ]:
# ============================================
# STEP 2: Import Libraries and Configure Settings
# ============================================
#
# WHAT WE'RE DOING:
# Loading all required modules and configuring the environment.
#
# WHY THIS MATTERS:
# - nest_asyncio.apply() fixes event loop issues in Colab/Jupyter
# - Settings.llm = None disables the LLM (we're only testing retrieval,
#   not generation, so we don't need an LLM for this notebook)
#
# WHAT YOU'LL SEE:
# A message about "MockLLM" - this is expected and harmless.
# It just means LlamaIndex is using a placeholder since we disabled the LLM.
# ============================================

import nest_asyncio
nest_asyncio.apply()

from llama_index.core import VectorStoreIndex, Document, Settings, get_response_synthesizer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.query_engine import RetrieverQueryEngine
import fitz  # PyMuPDF
import time

# Disable LLM - we're only comparing embedding models for retrieval
Settings.llm = None

# ============================================
# SECTION 2: Load the Pharmaceutical Document
# ============================================
#
# We'll extract text from a pharmaceutical SDF document
# to use as our test corpus for embedding comparison.
# ============================================

In [ ]:
# ============================================
# STEP 3: Load and Extract Text from PDF
# ============================================
#
# WHAT WE'RE DOING:
# Loading the pharmaceutical SDF document and extracting all text.
#
# WHY THIS MATTERS:
# This document contains product specifications, test methods,
# and quality control information - realistic content for
# testing pharmaceutical document retrieval.
#
# WHAT YOU'LL SEE:
# Word count from the extracted document.
# ============================================

# Load the pharmaceutical SDF document
pdf_path = "/content/sample-sdf-document.pdf"
doc = fitz.open(pdf_path)
text = "\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the pharmaceutical document.")

In [ ]:
# ============================================
# STEP 4: Split Document into Chunks
# ============================================
#
# WHAT WE'RE DOING:
# Breaking the document into overlapping chunks.
# We'll build the index separately for each embedding model.
#
# WHY THIS MATTERS:
# Chunking affects retrieval quality. Small chunks (50 tokens)
# with overlap (50 tokens) give precise results while maintaining context.
#
# WHAT YOU'LL SEE:
# Confirmation of chunk creation.
# ============================================

# Define a sentence splitter
text_splitter = SentenceSplitter(chunk_size=50, chunk_overlap=50)

# Turn raw text into a list of Document objects
documents = [Document(text=text)]

# Convert into nodes (smaller chunks)
nodes = text_splitter.get_nodes_from_documents(documents)

print(f"Created {len(nodes)} chunks from the document.")

# ============================================
# SECTION 3: Compare Embedding Models
# ============================================
#
# We'll test three popular open-source embedding models:
# 1. MiniLM-L6-v2 - Fast and lightweight (22M parameters)
# 2. BGE-small-en - Good balance of speed and quality (33M parameters)
# 3. E5-small-v2 - Strong semantic understanding (33M parameters)
# ============================================

In [ ]:
# ============================================
# STEP 5: Run Embedding Model Comparison
# ============================================
#
# WHAT WE'RE DOING:
# Testing each embedding model with the same pharmaceutical query
# and measuring retrieval time and quality.
#
# WHY THIS MATTERS:
# Different embedding models have different strengths:
# - MiniLM: Fastest, good for high-volume applications
# - BGE: Best accuracy for general text
# - E5: Strong semantic understanding, good for complex queries
#
# WHAT YOU'LL SEE:
# For each model: loading message, retrieval time, and top results.
# First run downloads models (~30-50MB each).
#
# IMPORTANT: We rebuild the index for each embedding model so that
# both the documents AND queries use the same embedding space.
# ============================================

embedding_models = {
    "MiniLM-L6-v2": "sentence-transformers/all-MiniLM-L6-v2",
    "BGE-small-en": "BAAI/bge-small-en-v1.5",
    "E5-small-v2": "intfloat/e5-small-v2"
}

# Pharmaceutical query about the SDF document
query = "What materials or components are included in this product?"

results = {}

for model_name, model_path in embedding_models.items():
    print(f"\n{'='*60}")
    print(f"Testing Embedding Model: {model_name}")
    print(f"{'='*60}")

    # Configure the embedding model
    embed_model = HuggingFaceEmbedding(model_name=model_path)
    Settings.embed_model = embed_model

    # Build a fresh index with this embedding model
    # This ensures documents are embedded with the same model as queries
    start_time = time.time()
    index = VectorStoreIndex(nodes)

    retriever = index.as_retriever(similarity_top_k=2)
    query_engine = RetrieverQueryEngine.from_args(retriever=retriever)

    # Run the query
    response = query_engine.query(query)
    end_time = time.time()

    # Store results
    results[model_name] = {
        "response": str(response),
        "time": round(end_time - start_time, 2)
    }

    print(f"Index build + retrieval time: {results[model_name]['time']} seconds")

In [ ]:
# ============================================
# STEP 6: Display Comparison Results
# ============================================
#
# WHAT WE'RE DOING:
# Displaying the results from each embedding model side by side.
#
# WHY THIS MATTERS:
# This helps you choose the right model for your use case:
# - Need speed? Choose the fastest model
# - Need accuracy? Choose the model with best results
# - Need balance? Compare time vs. quality tradeoffs
#
# WHAT YOU'LL SEE:
# For each model: total time (index build + retrieval) and the retrieved context.
# ============================================

print("\n" + "="*70)
print("EMBEDDING MODEL COMPARISON RESULTS")
print("="*70)

for model, result in results.items():
    print(f"\n{'='*60}")
    print(f"Model: {model}")
    print(f"{'='*60}")
    print(f"Index Build + Retrieval Time: {result['time']} seconds")
    print(f"\nRetrieved Context:")
    print("-" * 40)
    print(result['response'])
    print()

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print("\nModel Performance Ranking (by total time):")
sorted_results = sorted(results.items(), key=lambda x: x[1]['time'])
for i, (model, result) in enumerate(sorted_results, 1):
    print(f"  {i}. {model}: {result['time']}s")